In [1]:
from llama_cpp import Llama, llama_supports_gpu_offload 

print("GPU offload supported:", llama_supports_gpu_offload())

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 3060 Laptop GPU, compute capability 8.6, VMM: yes


GPU offload supported: True


In [2]:
# llm = Llama(
#         model_path="../llama.cpp/Qwen3-1.7B-Q8_0.gguf",
#         n_ctx=32768,
#         n_gpu_layers=-1             
#         )  


# llm_recipe1 = Llama(
#         model_path="../llama.cpp/Qwen3-1.7B-Q8_0.gguf",
#         lora_path="../llama.cpp/qwen3-1.7b-lora-recipe1-f16.gguf",
#         n_ctx=32768,
#         n_gpu_layers=-1             
#         )  

llm_recipe2 = Llama(
        model_path="../llama.cpp/Qwen3-1.7B-Q8_0.gguf",
        lora_path="../llama.cpp/qwen3-1.7b-lora-recipe2-f16.gguf",
        n_ctx=32768,
        n_gpu_layers=-1             
        )  

llama_model_load_from_file_impl: using device CUDA0 (NVIDIA GeForce RTX 3060 Laptop GPU) - 5120 MiB free
llama_model_loader: loaded meta data with 35 key-value pairs and 311 tensors from ../llama.cpp/Qwen3-1.7B-Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3 1.7B Hf
llama_model_loader: - kv   3:                           general.finetune str              = hf
llama_model_loader: - kv   4:                           general.basename str              = Qwen3
llama_model_loader: - kv   5:                         general.size_label str              = 1.7B
llama_model_loader: - kv   6:                            gener

In [3]:
SYSTEM_PROMPT_STRONG = """
You are a human-like math problem PLANNER ONLY. Do not solve. Do not compute/derive/simplify. No numeric results.

Task: extract ONLY what is necessary to reach the asked-for quantity <ANS>.
Omit anything not used downstream.

Output exactly one plain-text block:
<<<PLAN>>>
Goal: <ANS> = (what the problem asks, in words)
Unknowns:
- <U1>=...
- <U2>=...   
(only true unknowns / decision outcomes)
Core constraints:
- <C1>: ...
- <C2>: ...
Dependency skeleton:
- <R1> from <C?> and <U?>
- <R2> from <C?> and <R?>
Plan:
- Identify the minimum set of intermediates needed for <ANS>.
- Compute ... -> <R1>.
- Compute ... -> <R2>.
- Combine intermediates to express <ANS>. (still no execution)

<<<END>>>

Rules:
- Do NOT create placeholders for fixed givens; refer to them as “given in the statement”.
- Do NOT name standard sub-parameters unless they are required intermediates; use <Rk> instead.
- No equations or operator symbols; describe relations in words.
- Nothing outside the block.
- Do not produce a long plan and overdetailed plan.
"""

In [4]:
# sample 10 examples from train and validation sets deterministically and query the model
import random, json, os

max_tokens = 5000
# deterministic seed so sampling is repeatable
random.seed(76)

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_path = "finetune/350-50/train.jsonl"
val_path   = "finetune/350-50/validation.jsonl"

train_lines = read_jsonl(train_path)
val_lines   = read_jsonl(val_path)

n = 2
counter = 0
train_samples = train_lines
val_samples   = val_lines

output_file = "recipe2_sample_responses.jsonl"
with open(output_file, "w", encoding="utf-8") as out:
    for sample_type, samples in [("valid", val_samples)]:#, ("validation", val_samples)]:

        for sample in samples:
            try:
                entry = json.loads(sample)
                # find first user message
                user_text = None
                target_response= None
                for msg in entry.get("messages", []):
                    if msg.get("role") == "user":
                        user_text = msg.get("content", "")
                    if msg.get("role") == "assistant":
                        target_response = msg.get("content", "")
                        break
                if user_text is None:
                    user_text = sample  # fallback to raw line
            except Exception:
                user_text = sample

            # query the model using only the user prompt
            print(user_text)
            # query the model using only the user prompt
            while True:
                response = llm_recipe2.create_chat_completion(
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT_STRONG},
                        {"role": "user", "content": "/no_think" + user_text},
                    ],
                    temperature=0.00,
                    top_p=1.0,
                    top_k=0,
                    min_p=0.0,
                    max_tokens=max_tokens,
                    stop=["<|im_end|>"],
                )
                resp = response["choices"][0]["message"]["content"]
                # strip any <think> tags that may appear
                resp = resp.replace("<think>", "").replace("</think>", "")
                if len(resp) <= max_tokens:
                    counter+=1
                    break
                # too long, discard and pick a new sample if available
                print("response too long, resampling...")
                if samples:
                    sample = random.choice(samples)
                    print(sample)
                    continue
                else:
                    break

            # record user prompt separately from type
            record = {"type": sample_type, "user": user_text,"target_response":target_response, "response": resp}
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            print("------------------------------")
            if counter >= n:
                break

A cylindrical tank of radius 0.324 is initially empty and is filled with water. For the first 34 minutes, inflow is 13.08 L/min; for the next 23 minutes it is 64.94 L/min. Outflow is 6.47 L/min throughout. At the end of each minute (after applying that minute’s inflow and outflow), if the water height exceeds 0.447, a sloshing loss of 13.9% of the excess volume above height 0.447 occurs instantly. Find the final water height and the total lost volume.


llama_perf_context_print:        load time =     522.08 ms
llama_perf_context_print: prompt eval time =     521.25 ms /   424 tokens (    1.23 ms per token,   813.44 tokens per second)
llama_perf_context_print:        eval time =    8837.06 ms /   538 runs   (   16.43 ms per token,    60.88 tokens per second)
llama_perf_context_print:       total time =   10254.33 ms /   962 tokens
llama_perf_context_print:    graphs reused =        520
Llama.generate: 282 prefix-match hit, remaining 122 prompt tokens to eval


------------------------------
A drug concentration follows dC/dt = -0.231C between doses. You administer 7 identical bolus doses of size 37.22 at scheduled times t_i=(i-1)*3.74 hours for i=1..7. If concentration exceeds 44.51, you skip the next scheduled dose once. The planned total dose is exactly 260.54. Given k, H, and this schedule pattern, compute the final concentration at time 29.25 and how many doses were skipped.


llama_perf_context_print:        load time =     522.08 ms
llama_perf_context_print: prompt eval time =      49.42 ms /   122 tokens (    0.41 ms per token,  2468.44 tokens per second)
llama_perf_context_print:        eval time =    5676.17 ms /   329 runs   (   17.25 ms per token,    57.96 tokens per second)
llama_perf_context_print:       total time =    6279.91 ms /   451 tokens
llama_perf_context_print:    graphs reused =        318


------------------------------
